# COVID-19 Data Catalog
Convert the COVID-19 dataset from Snowflake Starschema into interactive HTML data catalog

In [1]:
# Read the metadata table from snowflake and convert it to HTML output

# Generate:
# List of valid tables (metadata table v. infoschema v. show tables)
# Metadata tables (for column descriptions)
# Data previews
# Data profiles

In [2]:
import snowflake.connector
import pandas as pd

from IPython.core.display import display, HTML
from pandas_profiling import ProfileReport
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline

import json

In [3]:
# Read connection configuration
home = str(Path.home()) + '/Documents/'
with open(home + 'snowflake.cfg') as f:
    config = f.readlines()

# Remove line returns    
config = [x.strip() for x in config]

# Set to local variables
sf_account = config[0]
sf_user = config[1]
sf_pwd = config[2]

In [4]:
# Connect to Snowflake
# https://uja84855.us-east-1.snowflakecomputing.com/console#/internal/worksheet

sf_warehouse = 'COMPUTE_WH'

ctx = snowflake.connector.connect(
    account = sf_account,
    user = sf_user,
    password = sf_pwd,
    warehouse = sf_warehouse
    # role = 'SYSADMIN'
    )
cs = ctx.cursor()

# Optional config when connecting ...
    #database = 'CIMBA_PROD',
    #schema = 'MCP'
# or after connecting ...
    # cs.execute("USE ROLE SYSADMIN;")
    # cs.execute("USE DATABASE CIMBA_PROD;")
    # cs.execute("USE WAREHOUSE CIMBA_WH;")

In [5]:
# Test the connection
print(type(ctx))
print(type(cs))

<class 'snowflake.connector.connection.SnowflakeConnection'>
<class 'snowflake.connector.cursor.SnowflakeCursor'>


# Get List of Tables

In [28]:
# Get metadata from this dedicated metadata table
table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."METADATA";'
df = pd.read_sql(table_query, ctx)

In [29]:
df.shape

(317, 7)

In [30]:
df.head(3)

,TABLE,DESCRIPTION,COLUMN,TYPE,NULLABLE,COMMENTS,SOURCE
0,CT_US_COVID_TESTS,US COVID-19 testing and mortality,None,None,None,None,None
1,CT_US_COVID_TESTS,ISO-3166-1 entity name,COUNTRY_REGION,VARCHAR(16777216),False,United States only,CovidTracking API
2,CT_US_COVID_TESTS,ISO-3166-2 entity name (U.S. State),PROVINCE_STATE,VARCHAR(16777216),True,United States only,CovidTracking API


In [9]:
lst_tablenames_meta = df['TABLE'].unique()
lst_tablenames_meta = sorted(lst_tablenames_meta)
print(len(lst_tablenames_meta))
lst_tablenames_meta

22


['CT_US_COVID_TESTS',
 'DEMOGRAPHICS',
 'HDX_ACAPS',
 'HS_BULK_DATA',
 'HUM_RESTRICTIONS_AIRLINE',
 'HUM_RESTRICTIONS_COUNTRY',
 'IHME_COVID_19',
 'JHU_COVID_19',
 'KFF_HCP_CAPACITY',
 'KFF_US_ICU_BEDS',
 'KFF_US_POLICY_ACTIONS',
 'KFF_US_STATE_MITIGATIONS',
 'METADATA',
 'NYT_US_COVID19',
 'PCM_DPS_COVID19',
 'PCM_DPS_COVID19_DETAILS',
 'RKI_GER_DETAILED_DASHBOARD',
 'SCS_BE_DETAILED_HOSPITALISATIONS',
 'SCS_BE_DETAILED_MORTALITY',
 'SCS_BE_DETAILED_PROVINCE_CASE_COUNTS',
 'SCS_BE_DETAILED_TESTS',
 'WHO_SITUATION_REPORTS']

In [10]:
# Get table information from info_schema
# select TABLE_NAME from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."TABLES" where TABLE_SCHEMA = 'PUBLIC';
table_query = 'select * from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."TABLES" where TABLE_SCHEMA = \'PUBLIC\';'
df_info_tables = pd.read_sql(table_query, ctx)
df_info_tables.head(3)

,TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,TABLE_OWNER,TABLE_TYPE,IS_TRANSIENT,CLUSTERING_KEY,ROW_COUNT,BYTES,RETENTION_TIME,...,USER_DEFINED_TYPE_CATALOG,USER_DEFINED_TYPE_SCHEMA,USER_DEFINED_TYPE_NAME,IS_INSERTABLE_INTO,IS_TYPED,COMMIT_ACTION,CREATED,LAST_ALTERED,AUTO_CLUSTERING_ON,COMMENT
0,STARSCHEMA_COVID19,PUBLIC,CT_US_COVID_TESTS,None,BASE TABLE,NO,None,8641,261120,1,...,None,None,None,YES,YES,None,2020-03-20 23:04:36.357000+00:00,2020-08-07 12:47:01.115000+00:00,NO,US COVID-19 testing and mortality
1,STARSCHEMA_COVID19,PUBLIC,DATABANK_DEMOGRAPHICS,None,BASE TABLE,NO,None,216,9216,1,...,None,None,None,YES,YES,None,2020-04-20 13:26:45.219000+00:00,2020-08-07 12:47:01.119000+00:00,NO,None
2,STARSCHEMA_COVID19,PUBLIC,DEMOGRAPHICS,None,BASE TABLE,NO,None,3140,126976,1,...,None,None,None,YES,YES,None,2020-03-26 19:26:14.091000+00:00,2020-08-07 12:47:01.119000+00:00,NO,"Demographic data, 2019"


In [11]:
# Get column information from info_schema
# select TABLE_NAME from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."TABLES" where TABLE_SCHEMA = 'PUBLIC';
table_query = 'select * from "STARSCHEMA_COVID19"."INFORMATION_SCHEMA"."COLUMNS" where TABLE_SCHEMA = \'PUBLIC\';'
df_info_cols = pd.read_sql(table_query, ctx)
df_info_cols.head(3)

,TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,COLUMN_NAME,ORDINAL_POSITION,COLUMN_DEFAULT,IS_NULLABLE,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,CHARACTER_OCTET_LENGTH,...,DTD_IDENTIFIER,IS_SELF_REFERENCING,IS_IDENTITY,IDENTITY_GENERATION,IDENTITY_START,IDENTITY_INCREMENT,IDENTITY_MAXIMUM,IDENTITY_MINIMUM,IDENTITY_CYCLE,COMMENT
0,STARSCHEMA_COVID19,PUBLIC,CT_US_COVID_TESTS,INICUCURRENTLYINCREASE,31,None,YES,NUMBER,NaN,NaN,...,None,NO,NO,None,None,None,None,None,None,Patients currently in ICU since previous date ...
1,STARSCHEMA_COVID19,PUBLIC,HUM_RESTRICTIONS_AIRLINE,LONG,3,None,YES,FLOAT,NaN,NaN,...,None,NO,NO,None,None,None,None,None,None,Indicative longitude of geography (centroid)
2,STARSCHEMA_COVID19,PUBLIC,PCM_DPS_COVID19_DETAILS,PROVINCE_STATE,2,None,YES,TEXT,16777216.0,16777216.0,...,None,NO,NO,None,None,None,None,None,None,ISO-3166-2 entity name


In [12]:
# Get a sorted list of table names
lst_tablenames_info = df_info_tables['TABLE_NAME'].unique()
lst_tablenames_info = sorted(lst_tablenames_info)
print(len(lst_tablenames_info))
lst_tablenames_info

26


['CT_US_COVID_TESTS',
 'DATABANK_DEMOGRAPHICS',
 'DEMOGRAPHICS',
 'GOOG_GLOBAL_MOBILITY_REPORT',
 'HDX_ACAPS',
 'HS_BULK_DATA',
 'HUM_RESTRICTIONS_AIRLINE',
 'HUM_RESTRICTIONS_COUNTRY',
 'JHU_COVID_19',
 'JHU_DASHBOARD_COVID_19_GLOBAL',
 'KFF_HCP_CAPACITY',
 'KFF_US_ICU_BEDS',
 'KFF_US_STATE_MITIGATIONS',
 'METADATA',
 'NYC_HEALTH_TESTS',
 'NYT_US_COVID19',
 'NYT_US_REOPEN_STATUS',
 'PCM_DPS_COVID19',
 'PCM_DPS_COVID19_DETAILS',
 'RKI_GER_COVID19_DASHBOARD',
 'SCS_BE_DETAILED_HOSPITALISATIONS',
 'SCS_BE_DETAILED_MORTALITY',
 'SCS_BE_DETAILED_PROVINCE_CASE_COUNTS',
 'SCS_BE_DETAILED_TESTS',
 'VH_CAN_DETAILED',
 'WHO_SITUATION_REPORTS']

In [13]:
table_toc = ''
for table in lst_tablenames_info:
    table_toc += '<p class="tablemenu" id="' + table + '" onclick="displayContent(this.id)">' + table + '</p>'
    table_toc += '\n'

In [14]:
# Output list to HTML snippet file
f_tables = open( "table_list.html", "w+")
f_tables.write(table_toc)
f_tables.close()


# Generate Overview for each Table

In [15]:
# TABLE_NAME
# TABLE_OWNER
# ROW_COUNT
# BYTES
# CREATED
# LAST_ALTERED
# COMMENT

In [16]:
def genDataTypeChart(data_types):
    # Get a count of the data types for this column and turn into JSON for a D3 chart
    # pass in data_types
    str_data = data_types.to_json(orient='table')

    # Modify the JSON names for d3
    str_data = str_data.replace('index', 'name')
    str_data = str_data.replace('DATA_TYPE', 'value')

    # Extract just the data keys and values from the JSON
    # Then convert JSON back to string for placement in HTML
    json_data = json.loads(str_data)
    data_vals = str(json_data['data'])

    # Need double quotes for final JSON string structure
    data_vals = data_vals.replace('\'', '\"')
    
    f_js = open( "js_temp_hbar_chart.txt", "r")
    js_temp = f_js.read().splitlines()
    f_js.close()
    
    # Remove start and end [] brackets
    #js_script = str(js_temp)[1:-1]
    
    js_script = '\n'.join(js_temp)
    
    js_final = '<script> var data = ' + data_vals + ';\n' + js_script + '</script>'
    
    return js_final

In [17]:
# Generate the HTML for the overview cards section for rows, columns and file size
def genTableCardOverview(num_rows, num_cols, num_bytes):

    num_rows = "{:,}".format(num_rows)
    num_rows = str(num_rows)
    
    num_cols = str(num_cols)

    num_bytes = "{:,}".format(num_bytes)
    num_bytes = str(num_bytes)

    ovw_crd_html = '\n'
    ovw_crd_html += '<div class="row">\n'
    ovw_crd_html += '  <div class="column">\n'
    ovw_crd_html += '    <div class="card">\n'
    ovw_crd_html += '      <p class="single_data_val">' + num_rows + '</p>\n'
    ovw_crd_html += '      <p class="single_data_val_title">Rows</p>\n'
    ovw_crd_html += '    </div>\n'
    ovw_crd_html += '  </div>\n'
    ovw_crd_html += '  <div class="column">\n'
    ovw_crd_html += '    <div class="card">\n'
    ovw_crd_html += '      <p class="single_data_val">' + num_cols + '</p>\n'
    ovw_crd_html += '      <p class="single_data_val_title">Columns</p>\n'
    ovw_crd_html += '    </div>\n'
    ovw_crd_html += '  </div>\n'
    ovw_crd_html += '  <div class="column">\n'
    ovw_crd_html += '    <div class="card">\n'
    ovw_crd_html += '      <p class="single_data_val">' + num_bytes + '</p>\n'
    ovw_crd_html += '      <p class="single_data_val_title">Bytes</p>\n'
    ovw_crd_html += '    </div>\n'
    ovw_crd_html += '  </div>\n'
    ovw_crd_html += '</div>\n'

    return ovw_crd_html


In [18]:
df_info_tables.shape

(26, 22)

In [19]:
for index, row in df_info_tables.iterrows():
    
    table = row['TABLE_NAME']
    print(table)
    
    tmp_html = ''
    tmp_html += '<!DOCTYPE html><html><head>'
    tmp_html += '<link rel="stylesheet" href="css/datacat.css">'
    tmp_html += '<link rel="stylesheet" href="css/tables.css">'
    tmp_html += '<meta name="viewport" content="width=device-width, initial-scale=1">'
    tmp_html += '<script src="https://d3js.org/d3.v3.min.js" charset="utf-8"></script>'
    tmp_html += '<link rel="stylesheet" href="css/datacat.css">'
    tmp_html += '</head>'
    tmp_html += '<body style="font-family:sans-serif;padding-left:20px;padding-right:20px;background-color:white">'
    tmp_html += '<h2>' + table + '</h2>'
    
    if row['COMMENT'] is not None:
        tmp_html += '<p><b>Description: </b>' + row['COMMENT'] + '</p>'
        
    tmp_html += '<p><b>Created: </b>' + str(row['CREATED']) + '</p>'
    tmp_html += '<p><b>Last Altered: </b>' + str(row['LAST_ALTERED']) + '</p>'
    
    
    # Get HTML for card overview section for rows, cols, byte size
    # =========================
    tmp_html += '<p><b>Summary:</b></p>'
    tmp_html += '<hr>'
    
    df_cols = df_info_cols[df_info_cols['TABLE_NAME'] == table]
    tmp_html += genTableCardOverview(row['ROW_COUNT'], df_cols.shape[0], row['BYTES'])
    
    # =========================
    
    tmp_html += '<br>'
    tmp_html += '<p><b>Column Data Types:</b></p>'
    # Insert horizontal bar chart here
    data_types = df_cols['DATA_TYPE'].value_counts()
    # df_dtypes = data_types.to_frame()
    # df_dtypes.sort_values(by = ['DATA_TYPE'], ascending=False)
    # tmp_html += df_dtypes.to_html()

    tmp_html += '<hr>'
    # Chart object
    tmp_html += '<div id="graphic"></div>'
    tmp_html += '<br>'
    tmp_html += '<hr>'
    
    # Add script for D3
    js_dtype_chart = genDataTypeChart(data_types)
    
    tmp_html += js_dtype_chart
    
    # =========================
    
    # Finish the HTML
    tmp_html += '</body></html>'
    
    # Write HTML to file
    # 
    f = open( "content/overview_" + table + ".html", "w+")
    f.write(tmp_html)
    f.close()
    
    

CT_US_COVID_TESTS
DATABANK_DEMOGRAPHICS
DEMOGRAPHICS
GOOG_GLOBAL_MOBILITY_REPORT
HDX_ACAPS
HS_BULK_DATA
HUM_RESTRICTIONS_AIRLINE
HUM_RESTRICTIONS_COUNTRY
JHU_COVID_19
JHU_DASHBOARD_COVID_19_GLOBAL
KFF_HCP_CAPACITY
KFF_US_ICU_BEDS
KFF_US_STATE_MITIGATIONS
METADATA
NYC_HEALTH_TESTS
NYT_US_COVID19
NYT_US_REOPEN_STATUS
PCM_DPS_COVID19
PCM_DPS_COVID19_DETAILS
RKI_GER_COVID19_DASHBOARD
SCS_BE_DETAILED_HOSPITALISATIONS
SCS_BE_DETAILED_MORTALITY
SCS_BE_DETAILED_PROVINCE_CASE_COUNTS
SCS_BE_DETAILED_TESTS
VH_CAN_DETAILED
WHO_SITUATION_REPORTS


In [26]:
lst_tablenames_meta

['CT_US_COVID_TESTS',
 'DEMOGRAPHICS',
 'HDX_ACAPS',
 'HS_BULK_DATA',
 'HUM_RESTRICTIONS_AIRLINE',
 'HUM_RESTRICTIONS_COUNTRY',
 'IHME_COVID_19',
 'JHU_COVID_19',
 'KFF_HCP_CAPACITY',
 'KFF_US_ICU_BEDS',
 'KFF_US_POLICY_ACTIONS',
 'KFF_US_STATE_MITIGATIONS',
 'METADATA',
 'NYT_US_COVID19',
 'PCM_DPS_COVID19',
 'PCM_DPS_COVID19_DETAILS',
 'RKI_GER_DETAILED_DASHBOARD',
 'SCS_BE_DETAILED_HOSPITALISATIONS',
 'SCS_BE_DETAILED_MORTALITY',
 'SCS_BE_DETAILED_PROVINCE_CASE_COUNTS',
 'SCS_BE_DETAILED_TESTS',
 'WHO_SITUATION_REPORTS']

# Generate Metadata (Column Descriptions)

In [31]:
# Create an HTML metadata summary for each table

# Prevent truncating of strings from df
pd.set_option("display.max_colwidth", 10000) 

# Build HTML summary for each table
for table in lst_tablenames_meta:
    # print (table)
    tmp_html = ''
    tmp_html += '<!DOCTYPE html><html><head>'
    tmp_html += '<link rel="stylesheet" href="css/datacat.css">'
    tmp_html += '<link rel="stylesheet" href="css/tables.css">'
    tmp_html += '<meta name="viewport" content="width=device-width, initial-scale=1">'
    tmp_html += '</head>'
    tmp_html += '<body style="font-family:sans-serif;padding-left:20px;padding-right:20px;background-color:white">'
    tmp_html += '<h2>' + table + '</h2>'
    
    df_temp = df.loc[(df['TABLE'] == table) & (df['COLUMN'].isnull())]

    # Table description
    tmp_html += (df_temp['DESCRIPTION'].to_string(index=False)).lstrip()
    tmp_html += '<br><br>'
    
    # Column metadata
    df_col = df.loc[(df['TABLE'] == table) & (df['COLUMN'].notnull())]
    df_col = df_col[['COLUMN', 'DESCRIPTION', 'COMMENTS', 'SOURCE']]
    df_col = pd.DataFrame(df_col, index=None)
    
    df_col.rename(columns = {'COLUMN':'COLUMN_NAME'}, inplace = True)
    tmp_html += df_col.to_html(justify='left', index=False, classes='pure-table pure-table-bordered')
    tmp_html += '<br><br>'
    tmp_html += '<hr>'
    
    # Finish the HTML
    tmp_html += '</body></html>'
    
    # Write HTML to file
    # 
    f = open( "content/metadata_" + table + ".html", "w+")
    f.write(tmp_html)
    f.close()

# Display to screen
# display(HTML(tmp_html))


# Generate Data Previews

In [21]:
# Create an HTML data preview for each table

# Prevent truncating of strings from df
pd.set_option("display.max_colwidth", 10000) 

# Build HTML summary for each table

for table in lst_tablenames_info:
    # print (table)
    tmp_html = ''
    tmp_html += '<!DOCTYPE html><html><head>'
    tmp_html += '<link rel="stylesheet" href="css/datacat.css">'
    tmp_html += '<link rel="stylesheet" href="css/tables.css">'
    tmp_html += '<meta name="viewport" content="width=device-width, initial-scale=1">'
    tmp_html += '</head>'
    tmp_html += '<body style="background-color:white;font-family:sans-serif;padding-left:20px;padding-right:20px">'
    tmp_html += '<h2>' + table + '</h2>'
    
    table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."' + table + '";'
    df_table = pd.read_sql(table_query, ctx)

    # First rows
    tmp_html += "<h3>First 5 Rows</h3>"
    df_temp = df_table.head(5)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')

    # Random sample of rows
    tmp_html += "<h3>Random 10 Rows</h3>"
    df_temp = df_table.sample(n=10)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')
    
    # Last rows
    tmp_html += "<h3>Last 5 Rows</h3>"
    df_temp = df_table.tail(5)
    tmp_html += df_temp.to_html(classes='pure-table pure-table-bordered')
    
    tmp_html += '<br><br>'
    tmp_html += '<hr>'
    
    # Finish the HTML
    tmp_html += '</body></html>'
    
    # Write HTML to file
    # 
    f = open( "content/data_preview_" + table + ".html", "w+")
    f.write(tmp_html)
    f.close()

# Generate Data Profiles

In [22]:
# This function generates the data profile report and saves it to an HTML file
# https://pandas-profiling.github.io/pandas-profiling/docs/master/rtd/pages/advanced_usage.html#variable-summary-settings

def generate_data_profile(df_table, table_name):
    
    rpt_title = 'Data Profile: ' + table_name.upper()
    
    #myreport = ProfileReport(df_table, title=rpt_title)
    # myreport = ProfileReport(df_table, title=rpt_title, minimal=True)
    # myreport = ProfileReport(df_table, title=rpt_title)

    myreport = ProfileReport(df_table, title=rpt_title,
                             
        vars={
          'num':{'chi_squared_threshold': 0},
          'cat':{'chi_squared_threshold':0}
      },

        missing_diagrams={
            "bar": True,
            "matrix": False,
            "heatmap": False,
            "dendrogram": False                    
        },

        correlations={
            "pearson":{
                "calculate": True,
                "warn_high_correlations": True,
                "threshold": 0.9
            } ,
            "spearman":{"calculate": False},
            "kendall":{"calculate": False},
            "phi_k":{"calculate": False},
            "cramers":{"calculate": False},
            "recoded":{"calculate": False}
        },

        samples={
            "head": 3,
            "tail": 3 
        }
    )
    
    filename = 'data_profile_' + table_name + ".html"
    myreport.to_file("content/" + filename)

In [23]:
# Create a data profile for each table and save to HTML

# Get list of tables
qry_tablenames = 'show tables in schema "STARSCHEMA_COVID19"."PUBLIC";'
results = cs.execute(qry_tablenames).fetchall()

i = 0
print('GENERATING DATA PROFILES for TABLES')
print('--------------------------')
for r in results:
    
    # NOTE: Exclude HS_BULK_DATA because too sparse
    if r[1] != "HS_BULK_DATA" or r[1] != "CT_US_COVID_TESTS":
        if i >= 30:
            exit()
        else:
            df = None
            sf_table = r[1]
            # NOTE:  Use limit or sample to control how much to retrieve
            #table_query = 'select * from "COVID19"."PUBLIC"."' + sf_table + '" + ' limit 10,000;'
            table_query = 'select * from "STARSCHEMA_COVID19"."PUBLIC"."' + sf_table + '" sample (100 rows);'

            df = pd.read_sql(table_query, ctx)
            # print(df.head(3))

            print(r[1] + ' ' + str(df.shape))

            generate_data_profile(df, sf_table)
        
            i += 1

print('--------------------------')
print('Total Table Profiles: ' + str(len(results)))

GENERATING DATA PROFILES for TABLES
--------------------------
CT_US_COVID_TESTS (100, 31)



DATABANK_DEMOGRAPHICS (100, 11)



DEMOGRAPHICS (100, 10)



GOOG_GLOBAL_MOBILITY_REPORT (100, 14)



HDX_ACAPS (100, 15)


...


SCS_BE_DETAILED_PROVINCE_CASE_COUNTS (100, 11)



SCS_BE_DETAILED_TESTS (100, 3)



VH_CAN_DETAILED (100, 9)



WHO_SITUATION_REPORTS (100, 14)



--------------------------
Total Table Profiles: 26
